# 🔮 Notebook 6: Predicciones con Nuevas Reviews

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ Usar modelos entrenados para **predecir** nuevas reviews
2. ✅ Escribir tus **propias reviews** y ver predicciones
3. ✅ Comparar predicciones de **diferentes modelos**
4. ✅ Analizar **casos difíciles**
5. ✅ Entender niveles de **confianza**

⏱️ **Tiempo estimado**: 20 minutos

---

## 💡 Concepto: Usar Modelos Ya Entrenados

**Entrenar** un modelo tarda tiempo (5-30 minutos).

**Usar** un modelo ya entrenado es instantáneo (< 1 segundo).

```python
# Una vez:
model.fit(X_train, y_train)  # Tarda 30 minutos
model.save('modelo.pkl')     # Guardar

# Muchas veces:
model = load('modelo.pkl')   # < 1 segundo
model.predict(nueva_review)  # < 1 segundo
```

---

## 🔧 Setup Inicial

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import os
import sys
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import text_preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer

print("✅ Setup completo!")
print("\n💡 Nota: Este notebook usa modelos pre-entrenados")
print("   Si no has entrenado modelos aún, algunos ejemplos no funcionarán")

## 🎯 Opción 1: Predictor Simple (Sin Modelos Entrenados)

Vamos a crear un predictor simple para demostración:

In [ ]:
# Función simple de predicción basada en palabras clave
def predict_simple(text):
    """
    Predictor simple basado en palabras positivas/negativas
    (Solo para demostración - no es ML real)
    """
    # Palabras positivas
    positive_words = [
        'excellent', 'great', 'amazing', 'wonderful', 'fantastic',
        'brilliant', 'superb', 'outstanding', 'masterpiece', 'love',
        'loved', 'perfect', 'best', 'incredible', 'awesome'
    ]
    
    # Palabras negativas
    negative_words = [
        'terrible', 'awful', 'horrible', 'bad', 'worst',
        'waste', 'boring', 'disappointing', 'hate', 'hated',
        'poor', 'pathetic', 'dull', 'garbage'
    ]
    
    # Convertir a minúsculas
    text_lower = text.lower()
    
    # Contar palabras
    pos_count = sum(1 for word in positive_words if word in text_lower)
    neg_count = sum(1 for word in negative_words if word in text_lower)
    
    # Calcular score
    score = (pos_count - neg_count + 5) / 10  # Normalizar a 0-1
    score = max(0, min(1, score))  # Limitar a [0, 1]
    
    # Predicción
    prediction = 1 if score >= 0.5 else 0
    label = "Positivo ✅" if prediction == 1 else "Negativo ❌"
    
    return {
        'prediction': prediction,
        'probability': score,
        'label': label,
        'confidence': abs(score - 0.5) * 2
    }

print("✅ Predictor simple creado")

## 🧪 Probar el Predictor Simple

In [ ]:
# Reviews de ejemplo
test_reviews = [
    "This movie is absolutely excellent! Best film I've ever seen!",
    "Terrible waste of time. Awful acting and boring plot.",
    "It was okay, nothing special but not terrible either.",
    "I loved every minute of it! Brilliant performance!",
    "Disappointing. Expected much better based on the reviews.",
    "Masterpiece! Outstanding cinematography and amazing story.",
    "Horrible movie. Worst film of the year.",
    "Great acting but the plot was a bit confusing."
]

print("🔮 PREDICCIONES:")
print("="*80)

results = []
for i, review in enumerate(test_reviews, 1):
    result = predict_simple(review)
    results.append(result)
    
    print(f"\n{i}. {result['label']} (confianza: {result['confidence']:.1%})")
    print(f"   Review: \"{review}\"")
    print(f"   Probabilidad: {result['probability']:.3f}")

print("\n" + "="*80)

## 🎯 Ejercicio Interactivo 1: Tus Propias Reviews

**Escribe tus propias reviews y ve las predicciones:**

In [ ]:
# 👇 ESCRIBE TU REVIEW AQUÍ:
mi_review = "This movie was amazing and I absolutely loved it!"

# Predecir
resultado = predict_simple(mi_review)

print("\n🔮 PREDICCIÓN DE TU REVIEW:")
print("="*70)
print(f"Review: \"{mi_review}\"")
print(f"\nPredicción: {resultado['label']}")
print(f"Probabilidad: {resultado['probability']:.3f}")
print(f"Confianza: {resultado['confidence']:.1%}")
print("="*70)

## 📊 Análisis de Casos Difíciles

Algunos textos son **difíciles de clasificar**:

In [ ]:
# Casos difíciles
difficult_cases = [
    "Good movie but not great",  # Mixto
    "It was fine",  # Neutral
    "Not bad",  # Negación doble
    "Could have been better",  # Implícito
    "I wanted to love it but couldn't",  # Contradictorio
]

print("🤔 CASOS DIFÍCILES:")
print("="*70)

for review in difficult_cases:
    result = predict_simple(review)
    print(f"\nReview: \"{review}\"")
    print(f"   Predicción: {result['label']}")
    print(f"   Confianza: {result['confidence']:.1%} ", end="")
    
    if result['confidence'] < 0.3:
        print("⚠️ BAJA CONFIANZA")
    else:
        print("")

print("\n" + "="*70)
print("💡 Observación: Reviews neutrales o mixtas tienen baja confianza")

## 🎨 Visualización: Distribución de Confianza

In [ ]:
import matplotlib.pyplot as plt

# Obtener confianzas
confidences = [r['confidence'] for r in results]
labels_viz = [r['label'] for r in results]

# Crear gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de confianza
axes[0].hist(confidences, bins=10, color='teal', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Confianza', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].set_title('Distribución de Confianza', fontsize=14, fontweight='bold')
axes[0].axvline(np.mean(confidences), color='red', linestyle='--', 
                label=f'Media: {np.mean(confidences):.2f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confianza por predicción
pos_conf = [c for c, l in zip(confidences, labels_viz) if l == "Positivo ✅"]
neg_conf = [c for c, l in zip(confidences, labels_viz) if l == "Negativo ❌"]

axes[1].boxplot([pos_conf, neg_conf], labels=['Positivo', 'Negativo'])
axes[1].set_ylabel('Confianza', fontsize=12)
axes[1].set_title('Confianza por Sentimiento', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 Ejercicio Interactivo 2: Batch Prediction

**Predice múltiples reviews a la vez:**

In [ ]:
# 👇 AGREGA TUS REVIEWS AQUÍ:
mis_reviews = [
    "Amazing movie, loved it!",
    "Waste of time, terrible",
    "Not bad but could be better",
    # Agrega más aquí...
]

# Predecir todas
print("\n🔮 PREDICCIONES EN LOTE:")
print("="*70)

for i, review in enumerate(mis_reviews, 1):
    result = predict_simple(review)
    print(f"\n{i}. {result['label']} ({result['confidence']:.0%} confianza)")
    print(f"   \"{review}\"")

print("\n" + "="*70)

## 📊 Tabla Comparativa de Resultados

In [ ]:
# Crear DataFrame con resultados
df_results = pd.DataFrame({
    'Review': test_reviews,
    'Predicción': [r['label'] for r in results],
    'Probabilidad': [f"{r['probability']:.3f}" for r in results],
    'Confianza': [f"{r['confidence']:.1%}" for r in results]
})

print("\n📊 TABLA DE RESULTADOS:")
print("="*100)
print(df_results.to_string(index=False))
print("="*100)

## 💡 Consejos para Interpretar Predicciones

### 🟢 Alta Confianza (> 70%)
- Predicción muy confiable
- Palabras clave fuertes presentes
- Sentimiento claro

### 🟡 Confianza Media (40-70%)
- Predicción moderada
- Review puede ser mixta
- Considerar contexto

### 🔴 Baja Confianza (< 40%)
- Review neutral o ambigua
- Sentimiento poco claro
- Puede requerir análisis manual

---

## 🎯 Desafío Final: Casos Complicados

**¿Puede el predictor manejar estos casos difíciles?**

In [ ]:
# Casos muy difíciles
challenging_reviews = [
    "This movie is not unwatchable",  # Negación doble
    "I expected worse",  # Implícito positivo
    "The acting was good but everything else was terrible",  # Mixto
    "Meh",  # Jerga neutral
    "I guess it was okay?",  # Incierto
]

print("\n🏆 DESAFÍO: CASOS MUY DIFÍCILES")
print("="*70)

for review in challenging_reviews:
    result = predict_simple(review)
    print(f"\nReview: \"{review}\"")
    print(f"   Predicción: {result['label']}")
    print(f"   Confianza: {result['confidence']:.1%}")
    
    # Análisis
    if result['confidence'] < 0.2:
        print(f"   ⚠️ DIFÍCIL: Modelo no está seguro")
    elif result['confidence'] < 0.5:
        print(f"   ⚡ MODERADO: Considerar contexto")
    else:
        print(f"   ✅ SEGURO: Alta confianza")

print("\n" + "="*70)

## 📊 Resumen Final

En este notebook aprendiste:

✅ **Usar modelos entrenados** para predicción instantánea
✅ **Predecir nuevas reviews** que el modelo nunca vio
✅ **Interpretar confianza** de las predicciones
✅ **Identificar casos difíciles** (neutrales, mixtos, ambiguos)
✅ **Análisis comparativo** de diferentes reviews

---

## 🎓 ¡Felicidades!

Has completado los **6 notebooks de Análisis de Sentimientos**:

1. ✅ Introducción y Dataset IMDB
2. ✅ Tokenización y Preprocesamiento
3. ✅ Feature Extraction (TF-IDF)
4. ✅ Modelos Clásicos (Naive Bayes, SVM)
5. ✅ Word Embeddings y LSTM
6. ✅ Predicciones Interactivas

### 🚀 Próximos Pasos:

1. **Experimentar** con diferentes parámetros
2. **Entrenar modelos** en el dataset completo
3. **Crear tu propia web app** con Streamlit
4. **Probar otros datasets** (productos, tweets, etc.)

**¡Feliz aprendizaje de NLP!** 🎉📚